In [0]:
# !pip install transformers datasets pandas torch torchvision torchaudio
# Cell 1 — HF Token Setup
import os
from huggingface_hub import login

# Use ONE option only:

# Option 1: Environment variable (preferred for orchestration)
os.environ["HF_TOKEN"] = "<YOUR_HF_TOKEN>"

# Option 2: Direct login (if not using env variable)
# login("<YOUR_HF_TOKEN>")


In [0]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import os

# Explicitly load GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2", token=os.environ.get("HF_TOKEN"))
model = AutoModelForCausalLM.from_pretrained("gpt2", token=os.environ.get("HF_TOKEN"))

# ✅ Text Generation pipeline without default max_length
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    clean_up_tokenization_spaces=False,
    # remove default max_length completely
    model_kwargs={"max_length": None}
)

# ✅ Table QA pipeline
qa = pipeline(
    "table-question-answering",
    model="google/tapas-base-finetuned-wtq",
    token=os.environ.get("HF_TOKEN")
)


In [0]:
# Cell 3 — Data Calling with HF integration

# Load dataset from Databricks table → Pandas
df = spark.read.table("samplesuperstore.bronzedata.orders").toPandas()

# Dataset preview
print("Initial dataset preview:")
print(df.head())

# Dataset shape
print("Dataset shape:", df.shape)

# Quick summary
print("Column names:", df.columns.tolist())
print("Missing values per column:\n", df.isnull().sum())


In [0]:
# Cell 4 — Data Cleaning

# Step 1: Handle negative profits
print("Negative profit count before cleaning:", (df["Profit"] < 0).sum())
df["Profit"] = df["Profit"].apply(lambda x: max(x, 0))
print("Negative profit count after cleaning:", (df["Profit"] < 0).sum())

# Step 2: Drop duplicates
print("Duplicates before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after cleaning:", df.duplicated().sum())

# Step 3: Handle missing values
print("Missing values per column before cleaning:\n", df.isnull().sum())
df = df.dropna()
print("Missing values per column after cleaning:\n", df.isnull().sum())

# Step 4: Standardize column names
df.columns = [col.strip().replace(" ", "_").lower() for col in df.columns]
print("Standardized column names:", df.columns.tolist())

# Step 5: Final dataset summary
print("Cleaned dataset shape:", df.shape)
print("Cleaned dataset preview:")
print(df.head())


In [0]:
# Cell 5 — Hugging Face Models integration

# ML demo function
def run_ml():
    print("Running ML (demo with text generation)...")
    output = generator(
        "Training ML model on Sample Super Store dataset",
        max_new_tokens=50,
        clean_up_tokenization_spaces=False   # suppress GPT-2 warning
    )[0]["generated_text"]
    return output

# Visualization demo function
def run_visualization(df):
    print("Running Visualization (QA demo)...")
    # Convert subset to string for TAPAS
    subset = df.head(20).astype(str)
    result = qa(table=subset, query="Which category has highest sales?")
    return result


In [0]:
# Cell 6 — Agent Executor (final version)

def agent_executor(df):
    # Step 1: anomaly check
    if (df["profit"] < 0).any():
        print("Cleaning triggered due to anomalies...")
        df["profit"] = df["profit"].apply(lambda x: max(x, 0))
        df = df.drop_duplicates().dropna()
    else:
        print("No anomalies → skipping Cleaning")

    # Step 2: ML demo (GPT-2 text generation)
    # ✅ Only max_new_tokens used, no max_length
    ml_output = generator(
        "Training ML model on Sample Super Store dataset",
        max_new_tokens=50
    )[0]["generated_text"]
    print("ML Output:\n", ml_output)

    # Step 3: Visualization demo (TAPAS QA)
    viz_output = run_visualization(df)
    print("Visualization Output:\n", viz_output)

    return "Dynamic orchestration completed"

# # Run orchestration
# result = agent_executor(df)
# print(result)


In [0]:
# Cell 7 — Run Agent Executor

result = agent_executor(df)
print(result)
